In [102]:
import pandas as pd
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from pathlib import Path
from typing import Literal
import pymupdf
import os
import re

In [103]:
FOLDER_PATH = "/Users/tomthomas/Documents/rag-foundation/data/papers"
paths = list(Path(FOLDER_PATH).glob("*.pdf"))

In [104]:
class Chunk(BaseModel):
    text: str
    chunk_id: str
    section_tag: Literal["References", "Content", "Appendix"]
    page_number: int = Field(ge=1)
    doc_id: str

class RetrievedChunk(BaseModel):
    chunk: Chunk
    score: float

class Citation(BaseModel):
    page_number: int = Field(ge=1)
    doc_id: str

class AnswerSentence(BaseModel):
    text: str
    citations: list[Citation]

class Answer(BaseModel):
    sentences: list[AnswerSentence]

In [105]:
def is_appendix_heading(lines: list[str], i: int) -> bool:
    s = lines[i].strip()
    if s.lower() == "appendix":
        return True
    if s == "A":
        for nxt in lines[i + 1:]:
            nxt = nxt.strip()
            if nxt:                      # first non-empty line after the "A"
                return nxt[0].isupper()
    return False

In [106]:
def load_pages(path: str) -> list[str]:
    """Returns list of pages from the given file path.
    
    Args:
        path (str): The file path to each paper.
    
    The page number of item i is i + 1 (pages are numbered from 1, matching the PDF page index).
    Returns:
        list: A list of pages from the given file path.
    
    """
    doc = pymupdf.open(path)
    pages = []
    for page in doc:
        pages.append(page.get_text())
    doc.close()
    return pages


In [107]:
def split_sections(pages: list[str]) -> dict[str, list[tuple[int, str]]]:
    """Returns a dictionary containing section and section pages.
        
        Args:
            pages (list[str]): list of pages from the document.

        Definition:
            A section is defined as a part of the document that starts with a section heading and ends before the next section heading.
            If no heading, then the section is by default content for the text.
            If reference is in middle of page, text after the reference is considered part of the reference section.
        
        Returns:
            dict: A dictionary containing section and section pages, where the key is the section name and the value is a list of tuples containing the page number and the text on that page.
        
        """
    sections = {
        "Content": [],
        "References": [],
        "Appendix": []
    }
    ref_found = False
    appendix_found = False
    for page_num, page in enumerate(pages, start=1):
        ref_text = ""
        appendix_text = ""
        content = ""
        lines = page.split("\n")
        for i, line in enumerate(lines):
            if ref_found and not is_appendix_heading(lines, i):
                ref_text += line + "\n"
            elif appendix_found and line.strip().lower() != "references":
                appendix_text += line + "\n"
            elif line.strip().lower() == "references":
                if appendix_found:
                    appendix_found = False
                ref_found = True
                ref_text += line + "\n"
            elif is_appendix_heading(lines, i) and ref_found:
                if ref_found:
                    ref_found = False
                appendix_found = True
                appendix_text += line + "\n"
            else:
                content += line + "\n"
        if content != "":
            sections["Content"].append((page_num, content))
        if ref_text != "":
            sections["References"].append((page_num, ref_text))
        if appendix_text != "":
            sections["Appendix"].append((page_num, appendix_text))
    return sections

In [ ]:
def clean_section(section: list[tuple[int, str]]) -> list[tuple[int, list[str]]]:
    """Cleans a section by removing any empty text entries/removing unwanted new line strings between sentences 
        or removing emtpy lines or removing unwanted data (for eg: lines with only digits or removing roman letters etc.).

        Args:
            section (list[tuple[int, str]]): A list of tuples containing the page number and the text on that page.

        Returns:
            list[tuple[int, list[str]]]: A cleaned list of tuples containing the page number and a list of paragraphs on that page.

    """

    cleaned_section = []
    roman_numeral = re.compile(
        r"(?i)M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})"
    )
    for page_number, text in section:
        sentences = []
        page_text = ""
        text = text.split("arXiv Template\nA PREPRINT\n")[-1]
        lines = text.split("\n")
        non_empty = [i for i, l in enumerate(lines) if l.strip()]
        if non_empty:
            first, last = non_empty[0], non_empty[-1]
        else:
            first, last = 0, len(lines) - 1
        for i, line in enumerate(lines):
            line = line.strip()
            if i == first and (line.isdigit() or line.startswith("Published as a conference paper at")):
                continue
            if i == last and line.isdigit():
                continue
            page_text += line + "\n"
        paragraph = re.split(r"(?<=\.)\n", page_text)
        for sentence in paragraph:
            sentence = sentence.strip()
            sentence = sentence.replace("-\n", "")
            idx = sentence.find("\n")
            if idx != -1:
                sentence = sentence.replace("\n", " ")
            if (sentence.isdigit() or sentence.isspace() or not sentence.strip() or sentence.startswith(("arXiv","Preprint","URL","org","doi")) or ("arXiv" in sentence) or (roman_numeral.fullmatch(sentence)) or (sentence.isalpha() and len(sentence) ==1)) and (len(sentence) < 50):
                continue
            sentences.append(sentence)
        if sentences:
            cleaned_section.append((page_number, sentences))
    return cleaned_section


In [113]:
def chunk_paragraphs(section: list[tuple[int, list[str]]], section_tag: str, doc_id: str, chunk_size: int = 600, chunk_overlap: int = 150) -> list[Chunk]:
    """Chunks a section paragraphs.

        Args:
            section (list[tuple[int, list[str]]]): A list of tuples containing the page number and a list of paragraphs on that page.
            section_tag (str): The tag or name of the section. Allowed values: ["References", "Content", "Appendix"]
            doc_id (str): The ID of the document.
            chunk_size (int, optional): The maximum size of each chunk in characters. Defaults to 600.
            chunk_overlap (int, optional): The number of characters to overlap between chunks. Defaults to 150.

        Paragraph upto chunk_size character is considered as a single chunk.
        If a paragraph short of chunk_size characters, it will be combined with the next paragraph to form a chunk of up to chunk_size characters, if the second paragraph can be added without breaking sentences.
        If a paragraph exceeds chunk_size characters, it will be split into multiple chunks, ensuring that each chunk ends at a sentence boundary and the succeding chunk starts with an overlap of chunk_overlap characters from preceeding chunk without breaking a sentence with a 10% character tolerance.
        if a chunk spans multiple pages, the page number will be recorded as the starting page.
        The chunking logic ensures that sentences are not broken and that overlaps are maintained for context continuity.
        Returns:
            list[Chunk]: A list of Chunk objects containing the chunked paragraphs, chunk id, section_tag, page number and doc_id.

    """
    flat = []
    for page_number, paragraphs in section:
        for paragraph in paragraphs:
            sentences = re.split(r"(?<!\b[A-Z]\.)(?<=[.!?])\s+(?=[A-Z])", paragraph)
            for sentence in sentences:
                if len(sentence) <= chunk_size:
                    flat.append((page_number, sentence))
                else:
                    words = sentence.split(" ")
                    current_chunk = ""
                    for word in words:
                        if len(current_chunk) + len(word) + 1 <= chunk_size:
                            current_chunk += word + " "
                        else:
                            if current_chunk:
                                flat.append((page_number, current_chunk.strip()))
                            current_chunk = word + " "
                    if current_chunk:
                        flat.append((page_number, current_chunk.strip()))
    buffer = []
    counter = 0
    chunks = []
    for page_number, sentence in flat:
        string = " ".join([s for _, s in buffer])
        if not buffer:
            size = len(string) + len(sentence)
        else:
            size = len(string) + len(sentence) + 1
        if size <= chunk_size:
            buffer.append((page_number, sentence))
        elif size > chunk_size and buffer:
            chunk_text = " ".join([s for _, s in buffer])
            chunk = Chunk(
                text=chunk_text,
                chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
                section_tag=section_tag,
                page_number=buffer[0][0],
                doc_id=doc_id
            )
            pg_no, txt = buffer[-1]
            buffer = []
            if len(txt) <= chunk_overlap * 1.10 and len(sentence + txt) + 1 <= chunk_size:
                buffer.append((pg_no, txt))
            counter += 1
            buffer.append((page_number, sentence))
            chunks.append(chunk)
    if buffer:
        chunk = Chunk(
            text=" ".join([s for _, s in buffer]),
            chunk_id=doc_id+f"_{section_tag}_{counter:04d}",
            section_tag=section_tag,
            page_number=buffer[0][0],
            doc_id=doc_id
        )
        chunks.append(chunk)
    return chunks

        

In [117]:
grand_total = 0
for path in paths:
    doc_id = path.stem
    pages = load_pages(path)
    sections = split_sections(pages)
    section_counts = {"Content": 0, "References": 0, "Appendix": 0}

    for section_name, section_pages in sections.items():
        cleaned_section = clean_section(section_pages)
        chunks = chunk_paragraphs(cleaned_section, section_name, doc_id)
        section_counts[section_name] = len(chunks)

    paper_total = sum(section_counts.values())
    grand_total += paper_total
    print(
        f"{doc_id}  content={section_counts['Content']}  "
        f"refs={section_counts['References']}  "
        f"appendix={section_counts['Appendix']}  total={paper_total}"
    )

print(f"Grand total: {grand_total}")

2603.10913  content=77  refs=32  appendix=93  total=202
2409.14924  content=161  refs=95  appendix=0  total=256
2510.12323  content=110  refs=8  appendix=38  total=156
2310.11511  content=98  refs=34  appendix=96  total=228
2601.17036  content=89  refs=15  appendix=39  total=143
2608.12875  content=77  refs=47  appendix=137  total=261
2505.03275  content=57  refs=7  appendix=0  total=64
2403.14403  content=100  refs=34  appendix=18  total=152
2601.12658  content=89  refs=19  appendix=0  total=108
2312.10997  content=164  refs=76  appendix=0  total=240
2401.15884  content=86  refs=26  appendix=22  total=134
2607.23121  content=92  refs=14  appendix=0  total=106
2608.27460  content=42  refs=10  appendix=0  total=52
2507.23386  content=79  refs=35  appendix=61  total=175
2604.05096  content=77  refs=22  appendix=44  total=143
2402.06196  content=344  refs=101  appendix=19  total=464
2407.16833  content=65  refs=21  appendix=20  total=106
2604.15484  content=98  refs=6  appendix=0  total=1

In [114]:
chunks = chunk_paragraphs(clean_section(sections["Appendix"]), "Appendix", doc_id)